# Ozon E-CUP 2026 — CatBoost V1

Первая версия решения:
- только human labels;
- 84 engineered numeric features;
- `category` как categorical feature;
- fuzzy similarity через `difflib.SequenceMatcher`;
- fixed holdout;
- Macro Average Precision по категориям;
- сохранение модели `catboost_v1.cbm`.

Исторический reference на исходном fixed holdout: **Macro PR-AUC ≈ 0.619731**.


In [ ]:
import os, re, gc, json, time, hashlib, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier, Pool
import torch

warnings.filterwarnings("ignore")

BASE="/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"
MATCHES_PATH=f"{BASE}/matches.parquet"
ITEMS_PATH=f"{BASE}/items_human.parquet"

FIXED_EVAL_PATH="/kaggle/working/ecup_fixed_eval_10pct.parquet"
FEATURE_CACHE_PATH="/kaggle/working/catboost_v1_features_20260818.parquet"
MODEL_PATH="/kaggle/working/catboost_v1.cbm"
PRED_PATH="/kaggle/working/catboost_v1_fixed_eval_predictions.parquet"

SEED=42
EXPECTED_FIXED_SIZE=36567
EXPECTED_FINGERPRINT="8edfff9275fe950015b7"
DEVICES="0:1" if torch.cuda.device_count()>=2 else "0"

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("CatBoost devices:", DEVICES)


## 1. Exact V1 feature engineering


In [ ]:
SPACE_RE=re.compile(r"\s+")
NON_ALNUM_RE=re.compile(r"[^0-9a-zа-я]+")
NUMBER_RE=re.compile(r"\d+(?:[.,]\d+)?")
TOKEN_RE=re.compile(r"[0-9a-zа-я]+")

SPECIAL_NAMES=["brand","model","article","manufacturer_article","oem","part_number","size","color","volume","weight","type"]

BASE_FEATURE_NAMES=[
"name_exact","name_fuzz_ratio","name_token_sort_ratio","name_token_set_ratio","name_token_jaccard","name_overlap_min","name_overlap_max",
"name_len1","name_len2","name_len_abs_diff","name_len_ratio","name_tokens1","name_tokens2","name_token_count_diff",
"numbers_both","numbers_exact","numbers_disjoint","numbers_jaccard","numbers_common_count","numbers_count1","numbers_count2",
"codes_both","codes_exact","codes_disjoint","codes_jaccard","codes_common_count",
"attr_count1","attr_count2","attr_common_keys","attr_union_keys","attr_key_jaccard","attr_equal_count","attr_conflict_count",
"attr_equal_ratio","attr_conflict_ratio","attr_numbers_both","attr_numbers_exact","attr_numbers_disjoint","attr_numbers_jaccard","attr_numbers_common_count"]

SPECIAL_FEATURE_NAMES=sum(([f"{x}_both_present",f"{x}_equal",f"{x}_conflict",f"{x}_similarity"] for x in SPECIAL_NAMES),[])
FEATURE_NAMES=BASE_FEATURE_NAMES+SPECIAL_FEATURE_NAMES
MODEL_FEATURES=["category"]+FEATURE_NAMES
assert len(FEATURE_NAMES)==84 and len(MODEL_FEATURES)==85

def normalize_text(x):
    if x is None: return ""
    x=str(x).lower().replace("ё","е").replace(",",".")
    return SPACE_RE.sub(" ",NON_ALNUM_RE.sub(" ",x)).strip()

def safe_json_loads(x):
    if not isinstance(x,str): return {}
    try: obj=json.loads(x)
    except: return {}
    if not isinstance(obj,dict): return {}
    out={}
    for k,v in obj.items():
        k,v=normalize_text(k),normalize_text(v)
        if k: out[k]=v
    return out

def fuzz_ratio(a,b): return SequenceMatcher(None,a,b).ratio()
def fuzz_token_sort(a,b): return fuzz_ratio(" ".join(sorted(a.split()))," ".join(sorted(b.split())))
def fuzz_token_set(a,b): return fuzz_ratio(" ".join(sorted(set(a.split())))," ".join(sorted(set(b.split()))))

def jaccard(a,b):
    if not a and not b: return 1.0
    u=a|b
    return len(a&b)/len(u) if u else 0.0

def overlap_min(a,b): return len(a&b)/min(len(a),len(b)) if a and b else 0.0
def overlap_max(a,b): return len(a&b)/max(len(a),len(b)) if a and b else 0.0
def get_numbers(text): return {x.replace(",",".") for x in NUMBER_RE.findall(text)}
def get_codes(text): return {t for t in TOKEN_RE.findall(text) if any(c.isdigit() for c in t) and any(c.isalpha() for c in t)}

def get_attribute_numbers(attrs):
    out=set()
    for v in attrs.values(): out|={x.replace(",",".") for x in NUMBER_RE.findall(v)}
    return out

def extract_special_attributes(attrs):
    r={x:"" for x in SPECIAL_NAMES}
    for k,v in attrs.items():
        if not v: continue
        rules={
            "brand": k=="бренд" or k=="brand" or "бренд товара" in k,
            "model": k=="модель" or k=="model" or "модель товара" in k,
            "manufacturer_article": "артикул производителя" in k or "manufacturer article" in k,
            "oem": k=="oem" or " oem" in f" {k}" or k.startswith("oem "),
            "part_number": "партномер" in k or "part number" in k or "partnumber" in k,
            "article": "артикул" in k and "производителя" not in k,
            "size": k=="размер" or "размер производителя" in k or "российский размер" in k or k=="size",
            "color": k in {"цвет","цвет товара","color"},
            "volume": k in {"объем","обьем","volume"} or "объем товара" in k,
            "weight": k in {"вес","вес товара","weight"},
            "type": k in {"тип","type","тип товара"},
        }
        for name,ok in rules.items():
            if ok and not r[name]: r[name]=v
    return r

def prepare_item(name,attributes):
    n=normalize_text(name); a=safe_json_loads(attributes)
    return {"name":n,"tokens":set(n.split()),"numbers":get_numbers(n),"codes":get_codes(n),
            "attrs":a,"attr_keys":set(a),"attr_numbers":get_attribute_numbers(a),
            "special":extract_special_attributes(a)}

def pair_features(x,y):
    n1,n2=x["name"],y["name"]; t1,t2=x["tokens"],y["tokens"]
    nm1,nm2=x["numbers"],y["numbers"]; c1,c2=x["codes"],y["codes"]
    a1,a2=x["attrs"],y["attrs"]; k1,k2=x["attr_keys"],y["attr_keys"]
    an1,an2=x["attr_numbers"],y["attr_numbers"]; s1,s2=x["special"],y["special"]
    l1,l2=len(n1),len(n2); nb=bool(nm1 and nm2); cb=bool(c1 and c2); ab=bool(an1 and an2)
    common,union=k1&k2,k1|k2; eq=sum(a1[k]==a2[k] for k in common); conf=len(common)-eq

    z=[
        float(n1==n2 and bool(n1)),fuzz_ratio(n1,n2),fuzz_token_sort(n1,n2),fuzz_token_set(n1,n2),
        jaccard(t1,t2),overlap_min(t1,t2),overlap_max(t1,t2),
        float(l1),float(l2),float(abs(l1-l2)),min(l1,l2)/max(l1,l2,1),
        float(len(t1)),float(len(t2)),float(abs(len(t1)-len(t2))),
        float(nb),float(nb and nm1==nm2),float(nb and len(nm1&nm2)==0),jaccard(nm1,nm2) if nb else 0.0,
        float(len(nm1&nm2)),float(len(nm1)),float(len(nm2)),
        float(cb),float(cb and c1==c2),float(cb and len(c1&c2)==0),jaccard(c1,c2) if cb else 0.0,float(len(c1&c2)),
        float(len(k1)),float(len(k2)),float(len(common)),float(len(union)),len(common)/len(union) if union else 0.0,
        float(eq),float(conf),eq/len(common) if common else 0.0,conf/len(common) if common else 0.0,
        float(ab),float(ab and an1==an2),float(ab and len(an1&an2)==0),jaccard(an1,an2) if ab else 0.0,float(len(an1&an2))
    ]

    for name in SPECIAL_NAMES:
        u,v=s1[name],s2[name]; both=bool(u and v)
        z += [float(both),float(both and u==v),float(both and u!=v),fuzz_ratio(u,v) if both else 0.0]

    return np.asarray(z,dtype=np.float32)


## 2. Load human data and fixed holdout


In [ ]:
matches=pd.read_parquet(MATCHES_PATH)
items=pd.read_parquet(ITEMS_PATH,columns=["id","name","category","attributes"])

item_index=pd.Index(items["id"])
idx1=item_index.get_indexer(matches["id1"])
idx2=item_index.get_indexer(matches["id2"])
if (idx1<0).any() or (idx2<0).any(): raise RuntimeError("Some ids are missing in items_human")

categories=items["category"].to_numpy()[idx1]

if os.path.exists(FIXED_EVAL_PATH):
    fixed=pd.read_parquet(FIXED_EVAL_PATH)
    fpairs=pd.MultiIndex.from_frame(fixed[["id1","id2"]])
    is_eval=pd.MultiIndex.from_frame(matches[["id1","id2"]]).isin(fpairs)
    print("Loaded existing fixed holdout:",len(fixed))
else:
    strata=pd.Series(categories).astype(str)+"__"+matches["target"].astype(str)
    _,eval_idx=train_test_split(np.arange(len(matches)),test_size=EXPECTED_FIXED_SIZE,random_state=SEED,stratify=strata)
    is_eval=np.zeros(len(matches),dtype=bool); is_eval[eval_idx]=True
    fixed=pd.DataFrame({"id1":matches.loc[is_eval,"id1"].to_numpy(),
                        "id2":matches.loc[is_eval,"id2"].to_numpy(),
                        "target":matches.loc[is_eval,"target"].to_numpy(),
                        "category":categories[is_eval]})
    fixed.to_parquet(FIXED_EVAL_PATH,index=False)
    print("Created fixed holdout:",len(fixed))

h=pd.util.hash_pandas_object(fixed[["id1","id2","target","category"]],index=False).values
fingerprint=hashlib.sha256(h.tobytes()).hexdigest()[:20]

print("pairs:",len(matches))
print("train:",int((~is_eval).sum()))
print("eval:",int(is_eval.sum()))
print("fingerprint:",fingerprint)
print("historical fingerprint:",EXPECTED_FINGERPRINT)


## 3. Build/load 84 V1 features


In [ ]:
cache_ok=False
if os.path.exists(FEATURE_CACHE_PATH):
    try:
        df=pd.read_parquet(FEATURE_CACHE_PATH)
        cache_ok=(len(df)==len(matches)
                  and np.array_equal(df["id1"].to_numpy(),matches["id1"].to_numpy())
                  and np.array_equal(df["id2"].to_numpy(),matches["id2"].to_numpy())
                  and all(c in df.columns for c in MODEL_FEATURES+["target","is_eval"]))
    except: cache_ok=False

if cache_ok:
    print("Loaded:",FEATURE_CACHE_PATH)
else:
    prepared=[prepare_item(n,a) for n,a in tqdm(zip(items["name"],items["attributes"]),total=len(items),desc="prepare items")]
    X=np.empty((len(matches),84),dtype=np.float32)
    for i in tqdm(range(len(matches)),desc="pair features"):
        X[i]=pair_features(prepared[idx1[i]],prepared[idx2[i]])

    df=pd.concat([
        pd.DataFrame({"id1":matches["id1"].to_numpy(),"id2":matches["id2"].to_numpy(),
                      "target":matches["target"].to_numpy(),"category":categories,"is_eval":is_eval}),
        pd.DataFrame(X,columns=FEATURE_NAMES)
    ],axis=1)

    df.to_parquet(FEATURE_CACHE_PATH,index=False)
    print("Saved:",FEATURE_CACHE_PATH)

assert np.isfinite(df[FEATURE_NAMES].to_numpy()).all()
print("feature table:",df.shape)


## 4. Inner validation


In [ ]:
train_df=df[~df["is_eval"]].reset_index(drop=True)
eval_df=df[df["is_eval"]].reset_index(drop=True)

strata=train_df["category"].astype(str)+"__"+train_df["target"].astype(str)
idx=np.arange(len(train_df))
itr,iva=train_test_split(idx,test_size=0.10,random_state=SEED,stratify=strata)
inner_train=train_df.iloc[itr]
inner_val=train_df.iloc[iva]

print("inner train:",len(inner_train))
print("inner val:",len(inner_val))
print("fixed eval:",len(eval_df))


## 5. Train CatBoost V1


In [ ]:
model=CatBoostClassifier(
    iterations=2000,depth=8,learning_rate=0.05,l2_leaf_reg=5,random_strength=1,
    loss_function="Logloss",eval_metric="Logloss",task_type="GPU",devices=DEVICES,
    border_count=128,random_seed=SEED,od_type="Iter",od_wait=150,
    allow_writing_files=False,verbose=200
)

t0=time.time()
model.fit(
    Pool(inner_train[MODEL_FEATURES],inner_train["target"],cat_features=["category"]),
    eval_set=Pool(inner_val[MODEL_FEATURES],inner_val["target"],cat_features=["category"]),
    use_best_model=True
)
best_iteration=model.get_best_iteration()
final_iterations=best_iteration+1
print("best iteration:",best_iteration)
print("final iterations:",final_iterations)
print(f"time: {time.time()-t0:.1f}s")


## 6. Competition metric


In [ ]:
def macro_ap_table(data,pred):
    z=data[["category","target"]].reset_index(drop=True).copy()
    z["predict"]=np.asarray(pred)
    rows=[]
    for c,p in z.groupby("category"):
        rows.append({"category":c,"pairs":len(p),"positive_rate":p["target"].mean(),
                     "PR_AUC":average_precision_score(p["target"],p["predict"])})
    table=pd.DataFrame(rows).sort_values("PR_AUC").reset_index(drop=True)
    return table["PR_AUC"].mean(),table

inner_pred=model.predict_proba(inner_val[MODEL_FEATURES])[:,1]
inner_macro,inner_table=macro_ap_table(inner_val,inner_pred)
print("Inner Macro PR-AUC:",inner_macro)
display(inner_table)


## 7. Retrain on all non-fixed human pairs


In [ ]:
final_model=CatBoostClassifier(
    iterations=final_iterations,depth=8,learning_rate=0.05,l2_leaf_reg=5,random_strength=1,
    loss_function="Logloss",task_type="GPU",devices=DEVICES,border_count=128,
    random_seed=SEED,allow_writing_files=False,verbose=200
)

final_model.fit(Pool(train_df[MODEL_FEATURES],train_df["target"],cat_features=["category"]))
final_model.save_model(MODEL_PATH)
print("Saved:",MODEL_PATH)


## 8. Final fixed-holdout evaluation


In [ ]:
eval_pred=final_model.predict_proba(eval_df[MODEL_FEATURES])[:,1]
macro,table=macro_ap_table(eval_df,eval_pred)
global_ap=average_precision_score(eval_df["target"],eval_pred)

pred_df=eval_df[["id1","id2","target","category"]].copy()
pred_df["predict"]=eval_pred
pred_df.to_parquet(PRED_PATH,index=False)

print("="*80)
print(f"Macro PR-AUC: {macro:.6f}")
print(f"Global AP:    {global_ap:.6f}")
print("Historical V1 reference: ~0.619731")
print("Model:",MODEL_PATH)
print("Predictions:",PRED_PATH)
print("="*80)
display(table)


## 9. Feature importance


In [ ]:
importance=pd.DataFrame({
    "feature":MODEL_FEATURES,
    "importance":final_model.get_feature_importance()
}).sort_values("importance",ascending=False).reset_index(drop=True)

display(importance.head(40))
